# Better Features for Cross-State ADHD Generalization
### CIS 631 Machine Learning Project – NSCH 2023–2025

**Research question.** Do better features, rather than more tuning, help a model trained in some U.S. states predict parent-reported ADHD in states it has never seen?

This notebook follows two earlier ones and does not repeat their negative findings:

* `ADHD_Cross_State_Generalization_Research 1 (2).ipynb` — linear SVM, 27 features. Published LOSO result: F1 0.566, ROC-AUC 0.860, precision 0.459.
* `ADHD_Cross_State_Generalization_Improved.ipynb` — validated threshold, histogram gradient boosting, feature set D. Published LOSO result: precision 0.601, recall 0.664, F1 0.631, ROC-AUC 0.886, PR-AUC 0.665. The pre-diagnosis set R reached only F1 0.368 and PR-AUC 0.30.

The Improved notebook already showed that hyperparameter search plateaued, answer-code encodings do not matter, ensembles do not help, and single extra items (developmental delay, speech, birth, individual ACE items) each added less than 0.005 PR-AUC. Any gain here has to come from new information, a better representation, or a cleaner early-signal set. If the new features do not help, this notebook says so.

**Ground rules, same as the earlier notebooks.**

* Same population: ages 6–17 with a valid yes/no answer on `k2q31a` (a parent’s report that a doctor ever said the child has ADHD). This is not a verified diagnosis.
* `SEED = 42`. The same 11 states from `GroupShuffleSplit` are untouched. Every design choice uses the other 40 development states. The keep/drop rule below is fixed before any result is read.
* Never used as inputs: state, survey year, survey weights, race, ethnicity, insurance, or any service/treatment item (special-education plan, ADHD medication, mental-health treatment, `k2q31b`/`c`/`d`, `addtreat`, and the other T5 items). No target encoding and no state-level statistics. Race and ethnicity are used only in the fairness check.
* Imputers, scalers, and age-band z-scores are fit inside training folds only.
* Results go to `output/nsch_features/` and do not write into `output/nsch_project/` or `output/nsch_improved/`.
* The target is a parent’s report. Nothing in this notebook is a diagnostic or screening tool.


## 1. Setup

**Method.** Fix the random seed, locate the topical Stata files, and prepare `output/nsch_features/`. Histogram gradient boosting is the tree model, matching the Improved notebook, so a change in score is a change in features. Logistic regression is the simple linear check. XGBoost is installed but not used. LightGBM is not installed. SHAP is installed, so the explanation section uses it alongside permutation importance.


In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from joblib import Parallel, delayed
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, GroupKFold, LeaveOneGroupOut,
                                     StratifiedKFold, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             confusion_matrix, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
N_JOBS = -1
T0 = time.time()

def _has(mod):
    try:
        __import__(mod)
        return True
    except Exception:
        return False

LIBS = {"xgboost": _has("xgboost"), "lightgbm": _has("lightgbm"), "shap": _has("shap")}
DATA_DIR = next(Path("data").glob("NSCH*"))
OUT = Path("output") / "nsch_features"
FIG = OUT / "figures"
MODELS_DIR = OUT / "models"
for p in (FIG, MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)
# This notebook must not touch the earlier result folders.
assert OUT.name == "nsch_features"
print("Data folder:", DATA_DIR)
print("Results folder:", OUT.resolve())
print("Libraries — XGBoost:", LIBS["xgboost"], "| LightGBM:", LIBS["lightgbm"], "| SHAP:", LIBS["shap"])
print("Model family for the ladder: HistGradientBoosting (LightGBM is absent; XGBoost is left unused so the model matches the Improved notebook).")


## 2. Load and Harmonize the Three Survey Years

**Method.** Load every topical Stata file in the data folder, stack the years, and keep children aged 6–17 with `k2q31a` equal to 1 (yes) or 2 (no). State codes are used only to split the data.

**Data note.** The zip already in this folder for 2023 is the screener file, which the original notebook correctly skipped. The Improved notebook was run on the topical files `nsch_2023e_topical.dta` (55,162 rows), `nsch_2024e_topical.dta` (51,375), and `nsch_2025e_topical.dta` (45,139). Those three topical files are what this notebook loads. The 2023 topical file is the Census public-use file (re-released December 2025 for an 8-case correction on `k4q01`, a variable this study does not use). The checks below confirm the row count, the analysis population, and the 11 untouched states. If any of those disagree with the Improved notebook, the notebook stops.


In [ ]:
frames = []
for path in sorted(DATA_DIR.glob("*.dta")):
    if "topical" not in path.name.lower():
        print("Skipping non-topical file:", path.name)
        continue
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    frames.append(df)
    print(f"Loaded {path.name}: {len(df):,} rows, {df.shape[1]} columns")
if len(frames) != 3:
    raise RuntimeError(f"Expected 3 topical files (2023, 2024, 2025); found {len(frames)}. The 2023 screener zip cannot stand in for the topical file.")
data = pd.concat(frames, ignore_index=True)
data["survey_year"] = data["year"].astype(int)
print("Combined:", data.shape, "| years:", sorted(data.survey_year.unique()))

labels = json.loads(Path("reference/nsch_topical_labels.json").read_text(encoding="utf-8"))["years"]["2025"]
DESC = {k.lower(): v for k, v in labels["descriptions"].items()}
print("Codebook descriptions:", len(DESC))


In [ ]:
TARGET = "k2q31a"
FIPS_TO_STATE = {1:"AL",2:"AK",4:"AZ",5:"AR",6:"CA",8:"CO",9:"CT",10:"DE",11:"DC",12:"FL",13:"GA",15:"HI",16:"ID",17:"IL",18:"IN",
                 19:"IA",20:"KS",21:"KY",22:"LA",23:"ME",24:"MD",25:"MA",26:"MI",27:"MN",28:"MS",29:"MO",30:"MT",31:"NE",32:"NV",
                 33:"NH",34:"NJ",35:"NM",36:"NY",37:"NC",38:"ND",39:"OH",40:"OK",41:"OR",42:"PA",44:"RI",45:"SC",46:"SD",47:"TN",
                 48:"TX",49:"UT",50:"VT",51:"VA",53:"WA",54:"WV",55:"WI",56:"WY"}
EXPECTED_TEST_STATES = ["AZ", "IA", "ID", "KY", "MO", "NH", "NJ", "NM", "SC", "TX", "VT"]
EXPECTED_N = 103353

kids = data[(data.sc_age_years >= 6) & (data[TARGET].isin([1, 2]))].copy()
kids["adhd"] = (kids[TARGET] == 1).astype(int)
kids["state"] = kids.fipsst.astype(int).map(FIPS_TO_STATE)
kids = kids.reset_index(drop=True)
if kids["state"].isna().any():
    raise RuntimeError("Unmapped FIPS codes: " + str(sorted(kids.loc[kids.state.isna(), "fipsst"].unique())))
print(f"Children aged 6–17 with a Yes/No ADHD answer: {len(kids):,} | states: {kids.state.nunique()}")
print(kids.groupby("survey_year").adhd.agg(children="size", adhd_pct=lambda s: round(100 * s.mean(), 2)))
if len(kids) != EXPECTED_N or kids.state.nunique() != 51:
    raise RuntimeError(f"Population does not match the Improved notebook (expected {EXPECTED_N} children and 51 states).")


## 3. Feature Discovery

**Method.** Read the 2025 codebook and the stacked data. A raw survey item is eligible when all of the following hold:

1. it is asked at ages 6–17 (not entirely missing in this age range),
2. it is missing for at most 10% of children in every year,
3. its non-missing answer codes are identical in 2023, 2024, and 2025 (true numbers such as age and the poverty ratio are exempt),
4. it is not an ID, a weight, or an imputation flag.

Every eligible item gets a timing tier from the question wording.

| Tier | Meaning |
|---|---|
| T0 fixed / birth | known at or soon after birth |
| T1 family background | household, hardship, neighborhood, caregiver background |
| T2 lifestyle now | current sleep, screen, activity, parent stress, parent–child connection |
| T3 behavior / school now | current behavior, school, friendships |
| T4 other diagnosis | another "ever told" condition |
| T5 services / treatment | exists because of a diagnosis; never a model input |

The families requested for this study are flourishing and self-regulation, sleep and routines, screen time and activity, parent stress and coping, parent–child connection, neighborhood safety and support, bullying and friendships, an ACE count, household hardship, adult 2, and general child health. Names are checked against the file. Items that fail the 10% rule are listed and are not used as raw inputs.


In [ ]:
# Locked feature lists from the two earlier notebooks. Set D is reconstructed, not re-searched.
F17 = ["sc_age_years", "sc_sex", "family_r", "totkids_r", "a1_grade", "a1_employed_r", "a1_menthealth",
       "ace1", "ace3", "ace8", "foodsit", "bedtime", "hoursleep", "screentime", "physactiv", "k8q11", "birthwt"]
EXTRA_10 = ["k7q70_r", "k7q83_r", "k7q84_r", "k7q85_r", "k7q04r_r", "grades", "k2q33a", "k2q32a", "k2q30a", "k2q35a"]
F27 = F17 + EXTRA_10
CANDIDATES = ["k2q34a", "k2q36a", "k2q37a", "k2q60a", "k2q38a", "k2q42a", "k2q05", "agepos4", "fpl_i1", "a1_physhealth",
              "ace4", "ace5", "ace6", "ace7", "ace9", "ace10", "ace11", "makefriend", "bully", "bullied_r", "repeated",
              "concussion", "k10q40_r", "k6q15"]
FOLLOW_UPS = ["k2q33b", "k2q33c", "k2q30b", "k2q30c", "k2q32b", "k2q32c", "k2q35b", "k2q35c", "k2q34b", "k2q34c"]
# Skip-pattern follow-ups: a blank means the parent said the child never had the condition, so the blank is 0.
SKIP_PARENT = {c: (c[:-1] + "a") for c in FOLLOW_UPS}
SKIP_PARENT.update({"k2q40b": "k2q40a", "k2q40c": "k2q40a", "diabetes_curr": "diabetes", "diabetes_desc": "diabetes"})

pre_miss = {}
for col, parent in SKIP_PARENT.items():
    if col not in kids.columns or parent not in kids.columns:
        print("Skip-recode column not in the file:", col, "parent", parent)
        continue
    pre_miss[col] = round(100 * kids[col].isna().mean(), 1)
    kids[col] = kids[col].where(kids[parent] != 2, 0)

BANNED_INPUTS = {
    "fipsst", "state", "year", "survey_year", "fwc", "stratum", "hhid",
    "sc_race_r", "sc_hispanic_r", "currins", "instype", "insgap", "currcov",
    "a1_born", "a2_born", "house_gen", "hhlanguage", "bornusa",
    "k2q31a", "k2q31b", "k2q31c", "k2q31d", "addtreat", "adhd",
    "k4q23", "k4q22_r", "sc_k2q22", "sc_cshcn", "k6q15", "k4q36", "k4q37", "k4q38",
    "sescurrsvc", "sesplanyr", "sesplanmo", "spcservmo", "ssidisability",
}
ID_OR_FLAG = {"fipsst", "stratum", "hhid", "year", "survey_year", "fwc", "tenure_if"}
TRUE_NUMBERS = {"sc_age_years", "fpl_i1", "a1_age", "a2_age", "k11q43r", "fwc"}

def code_tuple(series):
    vals = pd.Series(series.dropna().unique())
    if len(vals) == 0:
        return tuple()
    if pd.api.types.is_numeric_dtype(series):
        vals = np.round(vals.astype(float), 5)
    return tuple(sorted(vals.tolist()))

# Explicit tiers. Justification is the question wording plus why that wording falls in the tier.
TIER = {}
WHY = {}
def assign(cols, tier, why):
    for c in cols:
        TIER[c] = tier
        WHY[c] = why

assign(["sc_age_years", "sc_sex", "birthwt", "k2q05", "agepos4"], "T0",
       "Recorded at birth or a fixed demographic attribute, so it cannot be a consequence of a later diagnosis.")
assign(["family_r", "totkids_r", "a1_grade", "a1_employed_r", "a1_menthealth", "a1_physhealth",
        "a1_relation", "a1_marital", "a1_active", "a1_age", "a2_relation", "no_adult2",
        "ace1", "ace3", "ace4", "ace5", "ace6", "ace7", "ace8", "ace9", "ace10", "ace11", "ace_count",
        "foodsit", "fpl_i1", "k11q60", "k11q61", "k11q62", "k11q43r", "hardship_index",
        "k10q11", "k10q12", "k10q13", "k10q14", "k10q20", "k10q22", "k10q23", "k10q30", "k10q31",
        "k10q40_r", "neighborhood_score"], "T1",
       "Describes the household, caregiver, hardship, or neighborhood rather than the child's current symptoms or a service.")
assign(["bedtime", "hoursleep", "screentime", "physactiv", "k8q11", "sleep_routine_score",
        "k7q30", "k7q31", "k7q32", "k7q33", "k7q37", "k7q38",
        "k8q21", "k8q30", "k8q31", "k8q32", "k8q34", "k8q35", "parent_wellbeing", "connection_score",
        "hopeful", "strengths", "wktosolve", "talkabout", "goforhelp", "family_resilience", "k9q96",
        "k2q01", "k2q01_d", "cavities", "bmiclass"], "T2",
       "Describes current routines, parent coping, or overall health. ADHD or its treatment can change these, so they are not pure pre-diagnosis facts.")
assign(["k7q70_r", "k7q83_r", "k7q84_r", "k7q85_r", "k7q04r_r", "grades", "k7q82_r", "k7q02r_r",
        "makefriend", "bully", "bullied_r", "repeated", "k10q41_r", "k6q71_r", "memorycond",
        "self_reg_score", "emotional_score", "school_engagement", "behavior_score", "school_problems",
        "age_x_finish", "age_x_argue", "male_x_finish", "male_x_argue"], "T3",
       "Describes current behavior, school, or friendships. These may follow a diagnosis or its treatment.")
for z in ["k7q84_r_z", "k7q85_r_z", "k7q70_r_z", "k7q82_r_z", "k6q71_r_z", "self_reg_score_z"]:
    assign([z], "T3", "Age-band z-score of a current behavior item. Mean and standard deviation are fit on the training fold only.")
assign(["k2q33a", "k2q32a", "k2q30a", "k2q35a", "k2q34a", "k2q36a", "k2q37a", "k2q60a", "k2q38a", "k2q42a",
        "concussion", "k2q40a", "k2q40b", "k2q40c", "diabetes", "diabetes_curr", "diabetes_desc", "k2q43b",
        "overweight", "n_other_dx"] + FOLLOW_UPS, "T4",
       "Another 'ever told' condition, or a current/severity follow-up of one. It is often identified in the same evaluation as ADHD.")
assign(sorted(BANNED_INPUTS - {"state", "survey_year", "adhd", "fipsst", "year", "fwc", "stratum", "hhid"}), "T5",
       "Service, treatment, medication, or a variable the ground rules exclude from the model.")

# memorycond gets a more specific justification than the generic T3 line.
WHY["memorycond"] = ("The question asks if the child currently has serious difficulty concentrating, remembering, or making decisions. "
                     "That is a functional item, not a service, but it is close to the symptoms of ADHD, so a gain from it is recognition of current difficulty.")
WHY["k6q71_r"] = "Asks how often the child shows interest and curiosity. At ages 6–17 this is a flourishing item, current behavior rather than an early-life fact."
WHY["no_adult2"] = "Adult 2's relationship question uses code 8 for 'only one primary adult caregiver.' The health and education items for adult 2 are about 20% missing because of that skip, so they fail the 10% rule and are not used. This indicator keeps the skip itself."
WHY["ace_count"] = "Count of yes answers on ACE items ace3–ace11. ace1 (hard to cover basics) is kept in the hardship index instead, because it is a frequency item rather than a yes/no adversity."
WHY["ace10"] = "ACE item asking whether the child was treated unfairly because of race. It is the experience question, not the race category, which stays out of the model."
WHY["k8q31"] = "Asks how often the parent has felt the child is hard to care for. That is current parent aggravation and can be a reaction to the child's behavior."


In [ ]:
rows = []
for c in kids.columns:
    if c in ("adhd", "state"):
        continue
    miss = {y: 100 * kids.loc[kids.survey_year == y, c].isna().mean() for y in (2023, 2024, 2025)}
    codes = {y: code_tuple(kids.loc[kids.survey_year == y, c]) for y in (2023, 2024, 2025)}
    if c in TRUE_NUMBERS or c in ("ace_count", "hardship_index", "sleep_routine_score", "parent_wellbeing",
                                  "connection_score", "family_resilience", "neighborhood_score", "self_reg_score",
                                  "emotional_score", "school_engagement", "behavior_score", "school_problems",
                                  "n_other_dx", "no_adult2", "age_x_finish", "age_x_argue", "male_x_finish", "male_x_argue"):
        same = "numeric"
    elif codes[2023] == codes[2024] == codes[2025]:
        same = "yes"
    else:
        same = "NO"
    max_miss = max(miss.values())
    reason = ""
    if c.endswith("_if") or c in ID_OR_FLAG:
        reason = "ID, weight, or imputation flag"
    elif c in BANNED_INPUTS or TIER.get(c) == "T5":
        reason = "excluded input (identifier, fairness-only, insurance, or T5 service/treatment)"
    elif max_miss >= 99.5:
        reason = "not asked at ages 6–17 (essentially all missing)"
    elif max_miss > 10:
        reason = "missing for more than 10% of children aged 6–17"
    elif same == "NO":
        reason = "answer codes differ across 2023, 2024, and 2025"
    eligible = reason == ""
    family = ""
    rows.append({"feature": c, "description": DESC.get(c, ""), "tier": TIER.get(c, "unassigned"),
                 "justification": WHY.get(c, ""), "family": family,
                 **{f"missing_{y}": round(miss[y], 2) for y in (2023, 2024, 2025)},
                 "max_missing": round(max_miss, 2), "same_codes": same,
                 "eligible": eligible, "exclusion_reason": reason,
                 "skip_recode_missing_before": pre_miss.get(c, np.nan)})

catalog = pd.DataFrame(rows).set_index("feature")
print("Columns scanned:", len(catalog))
print(catalog.exclusion_reason.value_counts().to_string())
print("Eligible:", int(catalog.eligible.sum()), "| unassigned among eligible:", int(((catalog.eligible) & (catalog.tier == "unassigned")).sum()))


## 4. Feature Engineering

**Method.** Composites are row-wise functions of the answers. They do not use the target, the state, or the year. Diagnosis follow-ups were recoded above (blank when the parent said "never" becomes 0). `no_adult2` is the meaningful skip for a missing second caregiver.

Age-band z-scores are not stored as columns. A transformer learns the mean and standard deviation inside each age band (6–8, 9–11, 12–14, 15–17) on the training fold only, then applies those numbers to the test fold.

The interaction list is fixed in advance, not searched: age × finishes-tasks, age × argues, male × finishes-tasks, and male × argues.

Higher composite values mean more difficulty, less routine, or more hardship. Ordinal sums are missing if any piece is missing, so a missing answer is not treated as a zero.


In [ ]:
ACE_YES = ["ace3", "ace4", "ace5", "ace6", "ace7", "ace8", "ace9", "ace10", "ace11"]
eng = pd.DataFrame(index=kids.index)
eng["ace_count"] = (kids[ACE_YES] == 1).sum(axis=1).where(kids[ACE_YES].notna().any(axis=1))
hard_parts = pd.DataFrame({
    "ace1_often": (kids.ace1 >= 3).where(kids.ace1.notna()),
    "food_insec": (kids.foodsit >= 3).where(kids.foodsit.notna()),
    "cash": (kids.k11q60 == 1).where(kids.k11q60.notna()),
    "snap": (kids.k11q61 == 1).where(kids.k11q61.notna()),
    "meals": (kids.k11q62 == 1).where(kids.k11q62.notna()),
}).astype(float)
eng["hardship_index"] = hard_parts.sum(axis=1, min_count=1)
eng["no_adult2"] = np.where(kids.a2_relation.isna(), np.nan, (kids.a2_relation == 8).astype(float))
# bedtime: 1 always … 5 never. hoursleep: 1 = under 6 hours … 7 = 11 or more hours.
eng["sleep_routine_score"] = kids.bedtime + (8 - kids.hoursleep)
eng["parent_wellbeing"] = kids.a1_menthealth + kids.k8q30 + kids.k8q31 + kids.k8q32 + kids.k8q34
eng["connection_score"] = kids.k8q21 + (5 - kids.k8q11)
eng["family_resilience"] = kids.hopeful + kids.strengths + kids.wktosolve + kids.talkabout
disorder = pd.DataFrame({c: (kids[c] == 1).where(kids[c].notna()) for c in ["k10q20", "k10q22", "k10q23"]}).astype(float)
eng["neighborhood_score"] = disorder.sum(axis=1, min_count=1) + kids.k10q30 + kids.k10q31 + kids.k10q40_r
# Flourishing items are 1 always … 4 never, so a higher sum is less self-regulation.
eng["self_reg_score"] = kids.k7q84_r + kids.k7q85_r + kids.k6q71_r
eng["emotional_score"] = (5 - kids.k7q70_r) + kids.bully + kids.makefriend + kids.bullied_r
grades_for_sum = kids.grades.where(kids.grades <= 5)  # 6 = this school does not give letter grades
eng["school_engagement"] = kids.k7q82_r + kids.k7q83_r + grades_for_sum
# Same two summaries the Improved notebook built for set D.
OTHER_DX = ["k2q33a", "k2q32a", "k2q30a", "k2q35a", "k2q34a", "k2q36a", "k2q37a", "k2q60a"]
eng["n_other_dx"] = (kids[OTHER_DX] == 1).sum(axis=1)
eng["behavior_score"] = kids.k7q84_r + kids.k7q85_r + kids.k7q83_r + (5 - kids.k7q70_r)
# Same definition as the Improved notebook, including its treatment of a missing piece as not present.
eng["school_problems"] = ((kids.k7q04r_r >= 2).astype(float) + (kids.grades >= 4).astype(float) + (kids.repeated == 1).astype(float))
eng["age_x_finish"] = kids.sc_age_years * kids.k7q84_r
eng["age_x_argue"] = kids.sc_age_years * (5 - kids.k7q70_r)
eng["male_x_finish"] = (kids.sc_sex == 1).astype(float) * kids.k7q84_r
eng["male_x_argue"] = (kids.sc_sex == 1).astype(float) * (5 - kids.k7q70_r)
kids = pd.concat([kids, eng], axis=1)

ENGINEERED_OLD = ["n_other_dx", "behavior_score", "school_problems"]
COMPOSITES = ["ace_count", "hardship_index", "sleep_routine_score", "parent_wellbeing", "connection_score",
              "family_resilience", "neighborhood_score", "self_reg_score", "emotional_score", "school_engagement",
              "no_adult2"] + ENGINEERED_OLD
INTERACTIONS = ["age_x_finish", "age_x_argue", "male_x_finish", "male_x_argue"]
Z_FEATURES = ["k7q84_r_z", "k7q85_r_z", "k7q70_r_z", "k7q82_r_z", "k6q71_r_z", "self_reg_score_z"]
NUMERIC = ["sc_age_years", "fpl_i1", "a1_age", "k11q43r"] + COMPOSITES + INTERACTIONS + Z_FEATURES

eng_miss = (kids[COMPOSITES + INTERACTIONS].isna().mean() * 100).round(2).sort_values(ascending=False)
print("Missing % for engineered columns:")
print(eng_miss.to_string())
too_missing = [c for c in COMPOSITES + INTERACTIONS if kids[c].isna().mean() > 0.10]
if too_missing:
    print("NOT USED, missing > 10%:", too_missing)
else:
    print("Every engineered column is missing for at most 10% of children.")


In [ ]:
# Refresh catalog rows for columns created above, then attach the virtual z-score rows.
extra_rows = []
for c in COMPOSITES + INTERACTIONS:
    miss = {y: 100 * kids.loc[kids.survey_year == y, c].isna().mean() for y in (2023, 2024, 2025)}
    max_miss = max(miss.values())
    reason = "" if max_miss <= 10 else "engineered column missing for more than 10% of children"
    extra_rows.append({"feature": c, "description": WHY.get(c, "Engineered from survey answers, target not used"),
                       "tier": TIER.get(c, "unassigned"), "justification": WHY.get(c, ""), "family": "",
                       **{f"missing_{y}": round(miss[y], 2) for y in (2023, 2024, 2025)},
                       "max_missing": round(max_miss, 2), "same_codes": "numeric", "eligible": reason == "",
                       "exclusion_reason": reason, "skip_recode_missing_before": np.nan})
for c in Z_FEATURES:
    extra_rows.append({"feature": c, "description": "Age-band z-score, fit inside the training fold",
                       "tier": "T3", "justification": WHY[c], "family": "representation",
                       **{f"missing_{y}": 0.0 for y in (2023, 2024, 2025)},
                       "max_missing": 0.0, "same_codes": "numeric", "eligible": True,
                       "exclusion_reason": "", "skip_recode_missing_before": np.nan})
extra = pd.DataFrame(extra_rows).set_index("feature")
catalog = pd.concat([catalog.drop(index=extra.index.intersection(catalog.index), errors="ignore"), extra])
catalog = catalog[~catalog.index.duplicated(keep="last")]

def keep_eligible(cols, label):
    kept, dropped = [], []
    for c in cols:
        if c not in kids.columns and not str(c).endswith("_z"):
            dropped.append((c, "not in the data file"))
            continue
        if c not in catalog.index:
            dropped.append((c, "not in the catalog"))
            continue
        if not bool(catalog.loc[c, "eligible"]):
            dropped.append((c, catalog.loc[c, "exclusion_reason"]))
            continue
        if TIER.get(c, "").startswith("T5") or c in BANNED_INPUTS:
            dropped.append((c, "T5 or banned"))
            continue
        kept.append(c)
    if dropped:
        print(f"{label}: left out")
        for name, why in dropped:
            print(f"  {name}: {why}")
    return kept

# Requested families, verified against the file. Adult 2 health/education are expected to fail the 10% rule.
FAMILY_SPEC = {
    "flourishing": ["k6q71_r", "k7q84_r", "k7q85_r", "self_reg_score"],
    "sleep": ["bedtime", "hoursleep", "sleep_routine_score"],
    "screen_activity": ["screentime", "physactiv", "k7q30", "k7q31", "k7q32", "k7q33", "k7q37", "k7q38"],
    "parent_stress": ["k8q30", "k8q31", "k8q32", "k8q34", "a1_menthealth", "parent_wellbeing",
                      "hopeful", "strengths", "wktosolve", "talkabout", "goforhelp", "family_resilience"],
    "connection": ["k8q11", "k8q21", "k8q35", "k9q96", "connection_score"],
    "neighborhood": ["k10q11", "k10q12", "k10q13", "k10q14", "k10q20", "k10q22", "k10q23", "k10q30", "k10q31",
                     "k10q40_r", "k10q41_r", "neighborhood_score"],
    "bullying_friends": ["makefriend", "bully", "bullied_r", "emotional_score"],
    "ace": ["ace1", "ace3", "ace4", "ace5", "ace6", "ace7", "ace8", "ace9", "ace10", "ace11", "ace_count"],
    "hardship": ["ace1", "foodsit", "fpl_i1", "k11q60", "k11q61", "k11q62", "k11q43r", "hardship_index"],
    "adult2": ["a2_relation", "no_adult2", "a2_grade", "a2_menthealth", "a2_physhealth", "a2_age"],
    "general_health": ["k2q01", "k2q01_d", "cavities", "bmiclass", "overweight"],
    "school": ["grades", "k7q82_r", "k7q83_r", "k7q04r_r", "k7q02r_r", "repeated", "school_engagement", "school_problems"],
    "other_diagnosis": ["k2q33a", "k2q32a", "k2q30a", "k2q35a", "k2q34a", "k2q36a", "k2q37a", "k2q60a", "k2q38a",
                        "k2q42a", "concussion", "k2q40a", "k2q40b", "k2q40c", "diabetes", "diabetes_curr",
                        "diabetes_desc", "k2q43b", "n_other_dx"] + FOLLOW_UPS,
    "representation": INTERACTIONS + Z_FEATURES,
    "symptom_function": ["memorycond"],
}
for fam, cols in FAMILY_SPEC.items():
    for c in cols:
        if c in catalog.index and catalog.loc[c, "family"] == "":
            catalog.loc[c, "family"] = fam

print("Adult 2 and other requested items that fail eligibility (expected for the 20% skip and the age 0–5 items):")
watched = ["a2_grade", "a2_menthealth", "a2_physhealth", "a2_age", "a2_deplstat", "k3q25",
           "k6q70_r", "k6q72_r", "k6q73_r", "distracted", "hoursleep05", "k6q71_r", "memorycond"]
for c in watched:
    if c not in catalog.index:
        print(f"  {c}: not in the data")
    else:
        r = catalog.loc[c]
        print(f"  {c}: eligible={bool(r.eligible)} | max missing {r.max_missing}% | {r.exclusion_reason or r.tier} | {str(r.description)[:70]}")


In [ ]:
T0 = keep_eligible(["sc_age_years", "sc_sex", "birthwt", "k2q05", "agepos4"], "T0")
T1 = keep_eligible(["family_r", "totkids_r", "a1_grade", "a1_employed_r", "a1_menthealth", "a1_physhealth",
                    "a1_relation", "a1_marital", "a1_active", "ace1", "ace3", "ace4", "ace5", "ace6", "ace7",
                    "ace8", "ace9", "ace10", "ace11", "foodsit", "fpl_i1", "k11q60", "k11q61", "k11q62", "k11q43r",
                    "a2_relation", "no_adult2", "k10q11", "k10q12", "k10q13", "k10q14", "k10q20", "k10q22",
                    "k10q23", "k10q30", "k10q31", "k10q40_r"], "T1")
T2 = keep_eligible(["bedtime", "hoursleep", "screentime", "physactiv", "k8q11", "k7q30", "k7q31", "k7q32",
                    "k7q33", "k7q37", "k7q38", "k8q21", "k8q30", "k8q31", "k8q32", "k8q34", "k8q35", "k9q96",
                    "hopeful", "strengths", "wktosolve", "talkabout", "goforhelp", "k2q01", "k2q01_d",
                    "cavities", "bmiclass"], "T2")
T0T2_COMP = keep_eligible(["ace_count", "hardship_index", "sleep_routine_score", "parent_wellbeing",
                          "connection_score", "family_resilience", "neighborhood_score"], "T0–T2 composites")
T3 = keep_eligible(["k7q70_r", "k7q83_r", "k7q84_r", "k7q85_r", "k7q04r_r", "grades", "k7q82_r", "k7q02r_r",
                    "makefriend", "bully", "bullied_r", "repeated", "k10q41_r", "k6q71_r", "memorycond"], "T3")
T3_COMP = keep_eligible(["self_reg_score", "emotional_score", "school_engagement", "behavior_score", "school_problems"], "T3 composites")
T4 = keep_eligible(["k2q33a", "k2q32a", "k2q30a", "k2q35a", "k2q34a", "k2q36a", "k2q37a", "k2q60a", "k2q38a",
                    "k2q42a", "concussion", "k2q40a", "k2q40b", "k2q40c", "diabetes", "diabetes_curr",
                    "diabetes_desc", "k2q43b", "overweight", "n_other_dx"] + FOLLOW_UPS, "T4")
REP = keep_eligible(INTERACTIONS + Z_FEATURES, "representation")

def uniq(cols):
    return list(dict.fromkeys(cols))

E1 = uniq(T0 + T1 + T2 + T0T2_COMP)
E2 = uniq(E1 + T3 + T3_COMP)
N_FULL = uniq(E2 + T4 + REP)
# Set D, same construction and column order as the Improved notebook.
FD = sorted(set(F27 + [c for c in CANDIDATES if c != "k6q15"])) + FOLLOW_UPS + ENGINEERED_OLD
FR = ["sc_age_years", "sc_sex", "birthwt", "k2q05", "agepos4", "family_r", "totkids_r", "a1_grade",
      "a1_employed_r", "a1_menthealth", "fpl_i1", "foodsit", "ace1", "ace3", "ace8"]
N_RAW = [c for c in N_FULL if c not in set(COMPOSITES + INTERACTIONS + Z_FEATURES + ENGINEERED_OLD)]

for name, feats in {"L0": F27, "L1": FD, "R": FR, "E1": E1, "E2": E2, "N": N_FULL}.items():
    bad = [c for c in feats if c in BANNED_INPUTS or str(TIER.get(c, "")).startswith("T5")]
    missing_cols = [c for c in feats if not str(c).endswith("_z") and c not in kids.columns]
    if bad or missing_cols:
        raise RuntimeError(f"{name} has banned or missing columns: {bad or missing_cols}")

# E1 stops before behavior, school, and other diagnoses. E2 stops before other diagnoses.
e1_leak = [c for c in E1 if TIER.get(c) in ("T3", "T4", "T5")]
e2_leak = [c for c in E2 if TIER.get(c) in ("T4", "T5")]
if e1_leak or e2_leak:
    raise RuntimeError(f"Tier leak. E1: {e1_leak} | E2: {e2_leak}")
print(f"L0 {len(F27)} | L1 set D {len(FD)} | R {len(FR)} | E1 {len(E1)} | E2 {len(E2)} | N {len(N_FULL)} | N raw {len(N_RAW)}")

# New families are the columns in N that set D does not already contain, grouped for the keep rule.
new_cols = [c for c in N_FULL if c not in set(FD)]
NEW_FAMILIES = {}
for c in new_cols:
    fam = catalog.loc[c, "family"] if c in catalog.index and catalog.loc[c, "family"] else "other"
    NEW_FAMILIES.setdefault(fam, []).append(c)
print("New columns by family (not already in set D):")
for fam, cols in NEW_FAMILIES.items():
    print(f"  {fam} ({len(cols)}): {cols}")

catalog = catalog.copy()
catalog["used_in"] = ""
def _mark(feats, name):
    for c in feats:
        if c not in catalog.index:
            continue
        cur = catalog.loc[c, "used_in"]
        catalog.loc[c, "used_in"] = name if cur == "" else (cur if name in cur.split(",") else cur + "," + name)
_mark(F27, "L0")
_mark(FD, "L1")
_mark(FR, "R")
_mark(E1, "E1")
_mark(E2, "E2")
_mark(N_FULL, "N")
catalog.sort_values(["tier", "feature"]).to_csv(OUT / "feature_catalog.csv")
print("Wrote", OUT / "feature_catalog.csv")


## 5. Shared Tools, Splits, and the Nested Protocol

**Method.** The pipeline, the metrics, and the nested fold match the Improved notebook. For histogram gradient boosting the answer codes are passed through as numbers. Logistic regression and the linear SVM impute, one-hot encode the codes, and scale the true numbers. When a feature set contains an age-band z-score, that z-score is created inside the pipeline so the mean and standard deviation come from the training fold only.

The 11 geographic test states come from the same `GroupShuffleSplit` (test size 0.2, seed 42). Inner cross-validation on the training states chooses the gradient-boosting settings (the two settings from the Improved notebook) and the decision threshold. The threshold rule for the reported operating point is maximum F1. Precision at recall 0.70 is also reported from the precision–recall curve of the test scores, so that comparison does not depend on which threshold was chosen.

**Selection rule, fixed before looking at results.** On the 40 development states, with state-grouped 5-fold out-of-fold scores, a new family is kept only if its PR-AUC gain over set D is at least 0.005 in at least 4 of the 5 folds. Set D itself is locked; this notebook does not re-open the old feature search. The pre-specified rungs E1, E2, and N are still scored even when a family fails the rule.


In [ ]:
AGE_BANDS = [(6, 8), (9, 11), (12, 14), (15, 17)]

class FrameBuilder(BaseEstimator, TransformerMixin):
    """Select model columns and add age-band z-scores learned on the training rows only."""
    def __init__(self, features=None):
        # Stored as given. Copying here breaks sklearn.clone, which checks the object is unchanged.
        self.features = features
    def fit(self, X, y=None):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.z_stats_ = {}
        age = X["sc_age_years"] if "sc_age_years" in X.columns else None
        for feat in self.features:
            if not str(feat).endswith("_z"):
                continue
            base = feat[:-2]
            stats = {}
            for lo, hi in AGE_BANDS:
                m = (age >= lo) & (age <= hi)
                v = pd.to_numeric(X.loc[m, base], errors="coerce")
                mu = np.nanmean(v) if len(v) else np.nan
                sd = np.nanstd(v) if len(v) else np.nan
                if not np.isfinite(mu):
                    mu, sd = 0.0, 1.0
                if not np.isfinite(sd) or sd < 1e-6:
                    sd = 1.0
                stats[(lo, hi)] = (float(mu), float(sd))
            self.z_stats_[feat] = stats
        return self
    def transform(self, X):
        X = (X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)).copy()
        age = X["sc_age_years"]
        for feat, stats in self.z_stats_.items():
            base = pd.to_numeric(X[feat[:-2]], errors="coerce").to_numpy(dtype=float)
            out = np.full(len(X), np.nan)
            age_v = age.to_numpy()
            for (lo, hi), (mu, sd) in stats.items():
                m = (age_v >= lo) & (age_v <= hi)
                out[m] = (base[m] - mu) / sd
            X[feat] = out
        return X[self.features]

def columns_needed(features):
    need = []
    for f in features:
        if str(f).endswith("_z"):
            need += [f[:-2], "sc_age_years"]
        else:
            need.append(f)
    return list(dict.fromkeys(need))

def make_pipeline(features, model):
    features = list(features)
    has_z = any(str(c).endswith("_z") for c in features)
    if isinstance(model, HistGradientBoostingClassifier):
        if has_z:
            return Pipeline([("build", FrameBuilder(features)), ("model", model)])
        return Pipeline([("prep", ColumnTransformer([("codes", "passthrough", features)])), ("model", model)])
    num = [c for c in features if c in NUMERIC]
    cat = [c for c in features if c not in NUMERIC]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat)])
    steps = [("build", FrameBuilder(features))] if has_z else []
    return Pipeline(steps + [("prep", prep), ("model", model)])

MODEL_GRIDS = {
    "svm_bal": [dict(C=0.01)],
    "lr": [dict(C=1.0)],
    "hgb": [dict(learning_rate=0.05, max_leaf_nodes=15),
            dict(learning_rate=0.05, max_leaf_nodes=31, max_features=0.5, l2_regularization=0.1)],
}
def make_model(kind, params):
    if kind == "svm_bal":
        return LinearSVC(class_weight="balanced", max_iter=5000, **params)
    if kind == "lr":
        return LogisticRegression(max_iter=2000, **params)
    if kind == "hgb":
        return HistGradientBoostingClassifier(min_samples_leaf=100, max_iter=600, early_stopping=True,
                                              random_state=SEED, **params)
    raise ValueError(kind)

METHOD = {"svm_bal": "decision_function", "lr": "predict_proba", "hgb": "predict_proba"}
DEFAULT_THRESHOLD = {"svm_bal": 0.0, "lr": 0.5, "hgb": 0.5}
RULES = ["max_f1", "recall>=0.70"]

def get_scores(model, X):
    return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)

def metrics(y, pred, score, weights=None):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1], sample_weight=weights).ravel()
    prec = tp / (tp + fp) if (tp + fp) else np.nan
    rec = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else np.nan
    return {"Precision": prec, "Recall": rec, "F1": f1, "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
            "Balanced accuracy": (rec + spec) / 2, "ROC-AUC": roc_auc_score(y, score, sample_weight=weights),
            "PR-AUC": average_precision_score(y, score, sample_weight=weights),
            "Precision at recall 0.70": precision_at_recall(y, score, 0.70, weights),
            "Flagged %": 100 * (tp + fp) / (tp + fp + tn + fn), "TP": tp, "FP": fp, "FN": fn, "TN": tn}

def precision_at_recall(y, score, target=0.70, weights=None):
    prec, rec, _ = precision_recall_curve(y, score, sample_weight=weights)
    ok = np.where(rec[:-1] >= target)[0]
    if len(ok) == 0:
        return np.nan
    return float(prec[ok[-1]])

def choose_threshold(y, score, rule):
    prec, rec, thr = precision_recall_curve(y, score)
    prec, rec = prec[:-1], rec[:-1]
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    if rule == "max_f1" or len(rec) == 0:
        i = int(np.nanargmax(f1))
    else:
        target = float(rule.split(">=")[1])
        ok = np.where(rec >= target)[0]
        i = int(ok[np.nanargmax(prec[ok])]) if len(ok) else int(np.nanargmax(f1))
    return float(thr[i])

def nested_fold(train, test, feats, kind, inner_cv):
    y_tr, grid = train.adhd.values, MODEL_GRIDS[kind]
    Xtr, Xte = train[columns_needed(feats)], test[columns_needed(feats)]
    groups = train.state if isinstance(inner_cv, GroupKFold) else None
    oofs = []
    for params in grid:
        s = cross_val_predict(make_pipeline(feats, make_model(kind, params)), Xtr, y_tr,
                              groups=groups, cv=inner_cv, method=METHOD[kind])
        oofs.append(s[:, 1] if getattr(s, "ndim", 1) == 2 else s)
    best = int(np.argmax([average_precision_score(y_tr, s) for s in oofs]))
    thresholds = {"default": DEFAULT_THRESHOLD[kind], **{r: choose_threshold(y_tr, oofs[best], r) for r in RULES}}
    model = make_pipeline(feats, make_model(kind, grid[best])).fit(Xtr, y_tr)
    out = pd.DataFrame({"row": test.index, "score": get_scores(model, Xte), "params": str(grid[best])})
    for rule, t in thresholds.items():
        out["thr_" + rule] = t
    return out


In [ ]:
# Keep only the columns the experiments touch, so the parallel jobs do not ship the full survey file.
_keep = uniq(F27 + FD + FR + E1 + E2 + N_FULL + INTERACTIONS + COMPOSITES + [
    "k4q23", "adhd", "state", "survey_year", "fwc", "sc_sex", "sc_age_years", "sc_race_r", "sc_hispanic_r", "fpl_i1",
    "k7q84_r", "k7q85_r", "k7q70_r", "k7q82_r", "k6q71_r", "self_reg_score"])
kids = kids[[c for c in dict.fromkeys(_keep) if c in kids.columns]].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
_, test_idx = next(gss.split(kids, groups=kids.state))
TEST_STATES = sorted(kids.state.iloc[test_idx].unique())
if TEST_STATES != EXPECTED_TEST_STATES:
    raise RuntimeError("Untouched states do not match the Improved notebook.\\nGot " + str(TEST_STATES))
dev = kids[~kids.state.isin(TEST_STATES)].copy()
holdout = kids[kids.state.isin(TEST_STATES)].copy()
train_r, test_r = train_test_split(kids, test_size=0.2, stratify=kids.adhd, random_state=SEED)
early, late = kids[kids.survey_year <= 2024], kids[kids.survey_year == 2025]
PROTOCOLS = {
    "Random split": [(train_r, test_r)],
    "Geographic holdout": [(dev, holdout)],
    "Group K-Fold": [(kids.iloc[tr], kids.iloc[te]) for tr, te in GroupKFold(5).split(kids, groups=kids.state)],
    "LOSO": [(kids.iloc[tr], kids.iloc[te]) for tr, te in LeaveOneGroupOut().split(kids, groups=kids.state)],
    "Temporal (2023–24 → 2025)": [(early, late)],
    "Temporal + geographic": [(early[~early.state.isin(TEST_STATES)], late[late.state.isin(TEST_STATES)])],
}
INNER = {name: (StratifiedKFold(5, shuffle=True, random_state=SEED) if name == "Random split" else GroupKFold(5))
         for name in PROTOCOLS}
print("Untouched test states:", TEST_STATES)
print("Development children:", len(dev), "| holdout children:", len(holdout))
print({k: f"{len(v)} split(s)" for k, v in PROTOCOLS.items()})

DEV_FOLDS = np.empty(len(dev), dtype=int)
for i, (_, te) in enumerate(GroupKFold(5).split(dev, groups=dev.state)):
    DEV_FOLDS[te] = i


In [ ]:
FOLD_CACHE = {}

def run_many(pairs, grid_override=None):
    todo = {}
    for feats, kind, protocol in pairs:
        key = (tuple(feats), kind, protocol)
        if key not in FOLD_CACHE:
            todo[key] = (list(feats), kind, protocol)
    jobs = [(key, tr, te, feats, kind, protocol) for key, (feats, kind, protocol) in todo.items() for tr, te in PROTOCOLS[protocol]]
    if not jobs:
        return
    jobs.sort(key=lambda j: (j[4] != "hgb", -len(j[1])))
    print(f"Fitting {len(jobs)} train/test splits across {len(todo)} cached settings...", flush=True)
    t = time.time()
    outs = Parallel(n_jobs=N_JOBS)(delayed(nested_fold)(tr, te, feats, kind, INNER[protocol])
                                   for _, tr, te, feats, kind, protocol in jobs)
    for key in todo:
        parts = [o for j, o in zip(jobs, outs) if j[0] == key]
        FOLD_CACHE[key] = pd.concat(parts).set_index("row").join(kids[["adhd", "state", "survey_year", "fwc"]])
    print(f"  done in {time.time() - t:.0f}s", flush=True)

def cached_scores(feats, kind, protocol, rule):
    key = (tuple(feats), kind, protocol)
    if key not in FOLD_CACHE:
        run_many([(feats, kind, protocol)])
    p = FOLD_CACHE[key].copy()
    p["threshold"] = p["thr_" + rule]
    p["pred"] = (p.score >= p.threshold).astype(int)
    return p

def summarize_scores(p, weighted=False):
    return metrics(p.adhd, p.pred, p.score, p.fwc if weighted else None)

def dev_oof(feats, kind, params_index=0):
    params = MODEL_GRIDS[kind][params_index if kind != "hgb" else -1]
    est = make_pipeline(feats, make_model(kind, params))
    X = dev[columns_needed(feats)]
    s = cross_val_predict(est, X, dev.adhd, groups=dev.state, cv=GroupKFold(5), method=METHOD[kind])
    return s[:, 1] if getattr(s, "ndim", 1) == 2 else s

def fold_prauc(y, score, folds=DEV_FOLDS):
    return np.array([average_precision_score(y[folds == i], score[folds == i]) for i in range(5)])

def ranking_row(y, score):
    return {"PR-AUC": average_precision_score(y, score), "ROC-AUC": roc_auc_score(y, score),
            "Precision at recall 0.70": precision_at_recall(y, score, 0.70)}


## 6. Feature Ladder on the 40 Development States

**Method.** Every rung is scored with the same two model families: histogram gradient boosting (the setting that won the Improved notebook's search) and logistic regression. The scores are out-of-fold on the 40 development states, split by state. The rungs are:

* **L0** — the original 27 features
* **L1** — Improved set D
* **R** — the old pre-diagnosis set, T0–T1 only
* **E1** — early signal: T0–T2 plus the new T0–T2 composites, with no behavior, school, or other-diagnosis items
* **E2** — E1 plus T3 behavior and school items, still with no T4 diagnosis items
* **N** — E2 plus T4 items, follow-ups, and the engineered representation (composites already in E1/E2, plus age-band z-scores and the four interactions)
* **N raw** — the same raw survey items as N, without composites, z-scores, or interactions

A family ablation drops one new family from N at a time. The keep rule adds one new family to set D at a time and keeps it only if the PR-AUC gain is at least 0.005 in at least 4 of 5 folds.


In [ ]:
RUNGS = {
    "L0 original 27": F27,
    "L1 set D": FD,
    "R pre-diagnosis": FR,
    "E1 early signal": E1,
    "E2 behavior, no T4": E2,
    "N full new set": N_FULL,
    "N raw items only": N_RAW,
}
# Add-one family onto set D, and drop-one family from N.
ADD_SETS = {f"D + {fam}": uniq(FD + cols) for fam, cols in NEW_FAMILIES.items()}
DROP_SETS = {f"N − {fam}": [c for c in N_FULL if c not in set(cols)] for fam, cols in NEW_FAMILIES.items()}
LADDER_SETS = {**RUNGS, **ADD_SETS, **DROP_SETS}

def _one_oof(name, feats, kind):
    score = dev_oof(feats, kind)
    y = dev.adhd.to_numpy()
    folds = fold_prauc(y, score)
    row = {"set": name, "model": kind, "n_features": len(feats), **ranking_row(y, score)}
    for i, v in enumerate(folds, start=1):
        row[f"fold{i}"] = v
    return name, kind, score, row

ladder_jobs = [(name, feats, kind) for name, feats in LADDER_SETS.items() for kind in ("hgb", "lr")]
print(f"Development-state ladder: {len(ladder_jobs)} models", flush=True)
t = time.time()
ladder_out = Parallel(n_jobs=N_JOBS)(delayed(_one_oof)(name, feats, kind) for name, feats, kind in ladder_jobs)
print(f"Ladder finished in {time.time() - t:.0f}s", flush=True)
OOF = {(name, kind): score for name, kind, score, _ in ladder_out}
ladder = pd.DataFrame([row for *_, row in ladder_out])
ladder.to_csv(OUT / "ladder_dev.csv", index=False)
show = ladder[ladder["set"].isin(RUNGS)].copy()
show["set"] = pd.Categorical(show["set"], list(RUNGS), ordered=True)
show = show.sort_values(["set", "model"])
show.round(4)


In [ ]:
y_dev = dev.adhd.to_numpy()
base_folds = {kind: fold_prauc(y_dev, OOF[("L1 set D", kind)]) for kind in ("hgb", "lr")}
base_pool = {kind: average_precision_score(y_dev, OOF[("L1 set D", kind)]) for kind in ("hgb", "lr")}
add_rows = []
for fam in NEW_FAMILIES:
    for kind in ("hgb", "lr"):
        folds = fold_prauc(y_dev, OOF[(f"D + {fam}", kind)])
        delta = folds - base_folds[kind]
        add_rows.append({"family": fam, "model": kind, "n_new": len(NEW_FAMILIES[fam]),
                         "PR-AUC": average_precision_score(y_dev, OOF[(f"D + {fam}", kind)]),
                         "Δ PR-AUC": average_precision_score(y_dev, OOF[(f"D + {fam}", kind)]) - base_pool[kind],
                         "folds ≥ 0.005": int((delta >= 0.005).sum()),
                         "keep": bool((delta >= 0.005).sum() >= 4)})
add_table = pd.DataFrame(add_rows)
add_table.to_csv(OUT / "family_add_one.csv", index=False)
print("Keep rule (gain ≥ 0.005 PR-AUC in at least 4 of 5 development folds):")
add_table.round(4)


In [ ]:
# The decision uses histogram gradient boosting. Logistic regression is reported beside it.
kept_families = add_table[(add_table.model == "hgb") & (add_table.keep)].family.tolist()
KEPT_COLS = uniq([c for fam in kept_families for c in NEW_FAMILIES[fam]])
N_SELECTED = uniq(FD + KEPT_COLS)
print("Families kept:", kept_families if kept_families else "none")
print(f"N selected has {len(N_SELECTED)} features ({len(KEPT_COLS)} added to set D).")

n_folds = {kind: fold_prauc(y_dev, OOF[("N full new set", kind)]) for kind in ("hgb", "lr")}
drop_rows = []
for fam, cols in NEW_FAMILIES.items():
    for kind in ("hgb", "lr"):
        folds = fold_prauc(y_dev, OOF[(f"N − {fam}", kind)])
        delta = n_folds[kind] - folds  # positive => the family helps N
        drop_rows.append({"family": fam, "model": kind,
                          "Δ PR-AUC if removed": average_precision_score(y_dev, OOF[(f"N − {fam}", kind)]) - average_precision_score(y_dev, OOF[("N full new set", kind)]),
                          "folds where removal costs ≥ 0.005": int((delta >= 0.005).sum())})
drop_table = pd.DataFrame(drop_rows)
drop_table.to_csv(OUT / "family_ablation.csv", index=False)
drop_table.round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df = show[show.model == "hgb"].set_index("set")["PR-AUC"]
colors = ["gray", "darkorange", "slategray", "steelblue", "seagreen", "teal", "cadetblue"]
plot_df.plot.bar(ax=ax, color=colors[:len(plot_df)], rot=20)
ax.set_ylabel("PR-AUC on the 40 development states")
ax.set_title("Feature ladder, histogram gradient boosting, state-grouped out-of-fold scores")
ax.set_ylim(0.2, max(0.75, plot_df.max() + 0.05))
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "01_ladder_prauc.png", dpi=120)
plt.show()

hgb_add = add_table[add_table.model == "hgb"].set_index("family").sort_values("Δ PR-AUC")
fig, ax = plt.subplots(figsize=(8, 5))
hgb_add["Δ PR-AUC"].plot.barh(ax=ax, color=["seagreen" if k else "lightgray" for k in hgb_add["keep"]])
ax.axvline(0.005, color="black", ls="--", lw=1)
ax.set_xlabel("Δ PR-AUC when the family is added to set D")
ax.set_title("Green families clear the keep rule on gradient boosting")
plt.tight_layout()
plt.savefig(FIG / "02_family_add.png", dpi=120)
plt.show()
raw_vs = show[show["set"].isin(["N full new set", "N raw items only"])][["set", "model", "PR-AUC", "ROC-AUC", "Precision at recall 0.70"]]
raw_vs.round(4)


## 7. Nested Evaluation Under Every Protocol

**Method.** The sets that answer the research question are run through the same nested protocol as the Improved notebook: random split, geographic holdout of the 11 untouched states, group K-fold, leave-one-state-out, temporal (2023–24 → 2025), and temporal plus geographic. Inner state-grouped cross-validation on the training side picks the gradient-boosting settings and the max-F1 threshold. The original linear SVM on the 27 features, with its default threshold, is included so L0 can be checked against the published LOSO numbers. Set R is included so the early-signal rung can be compared with the old pre-diagnosis model under the same protocol. Logistic regression is the check on the development ladder above; the protocol table uses one model family so the differences are the features.


In [ ]:
EVAL_SETS = {
    "Original (SVM, 27, default threshold)": (F27, "svm_bal", "default"),
    "L0 (HistGB, 27, max-F1)": (F27, "hgb", "max_f1"),
    "L1 (HistGB, set D, max-F1)": (FD, "hgb", "max_f1"),
    "R (HistGB, pre-diagnosis, max-F1)": (FR, "hgb", "max_f1"),
    "E1 (HistGB, early signal, max-F1)": (E1, "hgb", "max_f1"),
    "E2 (HistGB, behavior no T4, max-F1)": (E2, "hgb", "max_f1"),
    "N (HistGB, full new set, max-F1)": (N_FULL, "hgb", "max_f1"),
}
if N_SELECTED != FD and N_SELECTED != N_FULL:
    EVAL_SETS["N selected (HistGB, families that passed, max-F1)"] = (N_SELECTED, "hgb", "max_f1")
elif N_SELECTED == FD:
    print("No new family passed the keep rule, so the selected model is set D and is not fit a second time.")
else:
    print("Every new family passed, so N selected is the full new set and is not fit a second time.")

FINAL_PROTOCOLS = ["Random split", "Geographic holdout", "Group K-Fold", "LOSO",
                   "Temporal (2023–24 → 2025)", "Temporal + geographic"]
run_many([(feats, kind, protocol) for feats, kind, _ in EVAL_SETS.values() for protocol in FINAL_PROTOCOLS])

rows = []
for name, (feats, kind, rule) in EVAL_SETS.items():
    for protocol in FINAL_PROTOCOLS:
        p = cached_scores(feats, kind, protocol, rule)
        rows.append({"Config": name, "Protocol": protocol, "Test children": len(p), **summarize_scores(p)})
        rows.append({"Config": name, "Protocol": protocol + " (survey-weighted)", "Test children": len(p),
                     **summarize_scores(p, weighted=True)})
results = pd.DataFrame(rows)
results.to_csv(OUT / "results_all_protocols.csv", index=False)
loso = results[(results.Protocol == "LOSO")].set_index("Config")
cols = ["Precision", "Recall", "F1", "FPR", "Balanced accuracy", "ROC-AUC", "PR-AUC", "Precision at recall 0.70", "Flagged %", "FP", "Test children"]
loso[cols].round(3)


## 8. Comparison with the Previous Notebooks

**Method.** The live comparison is the nested re-run above: original SVM, set D, the early-signal set, and the new set, under every protocol. As a cross-check, load `output/nsch_improved/results_all_protocols.csv` when that file is on disk. It is not in this folder (the Improved notebook was executed elsewhere and only its printed tables are stored in the `.ipynb`), so the cross-check also reads the LOSO table printed in that executed notebook and compares it with this re-run. A per-state LOSO chart shows where the new scores are higher or lower. Paired state bootstrap intervals (1,000 resamples of the 51 states) are reported for the change in F1 and PR-AUC.


In [ ]:
def parse_published_loso():
    path = Path("output/nsch_improved/results_all_protocols.csv")
    if path.exists():
        prev = pd.read_csv(path)
        prev = prev[prev.Protocol == "LOSO"][["Config", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]]
        prev["source"] = "output/nsch_improved/results_all_protocols.csv"
        return prev
    nb_path = Path("ADHD_Cross_State_Generalization_Improved.ipynb")
    published = {
        "Original (SVM, 27 feat., default threshold)": (0.459, 0.738, 0.566, 0.860, 0.591),
        "Final improved (HistGB, set D, max-F1 threshold)": (0.601, 0.664, 0.631, 0.886, 0.665),
        "Pre-diagnosis model (HistGB, set R)": (0.268, 0.586, 0.368, 0.683, 0.304),
    }
    # Confirm those three rows are actually in the executed notebook output before using them.
    text = nb_path.read_text(encoding="utf-8")
    missing = [k for k in published if k not in text]
    if missing:
        print("Could not find published rows in the Improved notebook:", missing)
    rows = [{"Config": k, "Precision": v[0], "Recall": v[1], "F1": v[2], "ROC-AUC": v[3], "PR-AUC": v[4],
             "source": "printed LOSO table in the executed Improved notebook"} for k, v in published.items()]
    return pd.DataFrame(rows)

published = parse_published_loso()
our = results[results.Protocol == "LOSO"].set_index("Config")
cross = pd.DataFrame({
    "This notebook, original SVM": our.loc["Original (SVM, 27, default threshold)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
    "Published original SVM": published.set_index("Config").loc["Original (SVM, 27 feat., default threshold)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
    "This notebook, set D": our.loc["L1 (HistGB, set D, max-F1)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
    "Published set D": published.set_index("Config").loc["Final improved (HistGB, set D, max-F1 threshold)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
    "This notebook, set R": our.loc["R (HistGB, pre-diagnosis, max-F1)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
    "Published set R": published.set_index("Config").loc["Pre-diagnosis model (HistGB, set R)", ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
}).T
cross["source note"] = published["source"].iloc[0]
cross.to_csv(OUT / "crosscheck_previous_loso.csv")
print("Cross-check source:", published["source"].iloc[0])
cross.round(3)


In [ ]:
# Wide comparison: original, set D, early signal, full new set, and the selected set when it differs.
compare_names = [c for c in ["Original (SVM, 27, default threshold)", "L0 (HistGB, 27, max-F1)",
                             "L1 (HistGB, set D, max-F1)", "R (HistGB, pre-diagnosis, max-F1)",
                             "E1 (HistGB, early signal, max-F1)", "E2 (HistGB, behavior no T4, max-F1)",
                             "N (HistGB, full new set, max-F1)", "N selected (HistGB, families that passed, max-F1)"]
                 if c in set(results.Config)]
unweighted = results[results.Protocol.isin(FINAL_PROTOCOLS) & results.Config.isin(compare_names)]
for metric in ["F1", "PR-AUC", "Precision at recall 0.70", "Precision"]:
    pivot = unweighted.pivot(index="Config", columns="Protocol", values=metric).loc[compare_names, FINAL_PROTOCOLS]
    pivot.to_csv(OUT / f"compare_{metric.replace(' ', '_').replace('.', '')}.csv")
f1_pivot = unweighted.pivot(index="Config", columns="Protocol", values="F1").loc[compare_names, FINAL_PROTOCOLS]
fig, ax = plt.subplots(figsize=(12, 4.5))
f1_pivot.T.plot.bar(ax=ax, rot=20)
ax.set_ylabel("F1")
ax.set_title("F1 under every protocol (max-F1 threshold chosen inside training folds)")
ax.legend(fontsize=7, loc="lower right")
ax.set_ylim(0, 0.85)
plt.tight_layout()
plt.savefig(FIG / "03_protocol_f1.png", dpi=120)
plt.show()
f1_pivot.round(3)


In [ ]:
def by_state(name):
    feats, kind, rule = EVAL_SETS[name]
    p = cached_scores(feats, kind, "LOSO", rule)
    rows = []
    for state, g in p.groupby("state"):
        m = metrics(g.adhd, g.pred, g.score)
        rows.append({"state": state, "children": len(g), "ADHD %": 100 * g.adhd.mean(),
                     "Precision": m["Precision"], "Recall": m["Recall"], "F1": m["F1"],
                     "PR-AUC": m["PR-AUC"], "Precision at recall 0.70": m["Precision at recall 0.70"]})
    return pd.DataFrame(rows).set_index("state")

NEW_NAME = "N selected (HistGB, families that passed, max-F1)" if "N selected (HistGB, families that passed, max-F1)" in EVAL_SETS else "N (HistGB, full new set, max-F1)"
# If nothing was added, the honest "new" model is set D. The chart then compares the full pre-specified set N with set D,
# and the text says the keep rule did not adopt it.
CHART_NEW = "N (HistGB, full new set, max-F1)"
st_orig, st_d, st_new, st_e1 = (by_state("Original (SVM, 27, default threshold)"), by_state("L1 (HistGB, set D, max-F1)"),
                                by_state(CHART_NEW), by_state("E1 (HistGB, early signal, max-F1)"))
state_table = pd.concat([st_orig.add_prefix("orig "), st_d.add_prefix("D "), st_new.add_prefix("N "), st_e1.add_prefix("E1 ")], axis=1)
state_table["Δ F1 N−D"] = st_new.F1 - st_d.F1
state_table["Δ PR-AUC N−D"] = st_new["PR-AUC"] - st_d["PR-AUC"]
state_table.to_csv(OUT / "state_results_loso.csv")
LOW_ADHD = ["NV", "NM", "CA", "HI", "WY"]
print("ADHD prevalence in the states flagged in the Improved notebook:", {s: round(st_d.loc[s, "ADHD %"], 1) for s in LOW_ADHD})
print("States where N F1 is lower than set D:", list(state_table.index[state_table["Δ F1 N−D"] < 0]))

order = state_table["Δ F1 N−D"].sort_values().index
fig, ax = plt.subplots(figsize=(16, 4.5))
delta = state_table.loc[order, "Δ F1 N−D"]
ax.bar(np.arange(len(order)), delta, color=["indianred" if s in LOW_ADHD else ("seagreen" if v >= 0 else "steelblue") for s, v in delta.items()])
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=90)
ax.set_ylabel("Δ F1 (full new set − set D)")
ax.set_title("LOSO: where the full new feature set is better or worse than set D (red = low-ADHD states NV, NM, CA, HI, WY)")
plt.tight_layout()
plt.savefig(FIG / "04_state_delta_f1.png", dpi=120)
plt.show()
state_table.loc[LOW_ADHD, ["D ADHD %", "D F1", "N F1", "Δ F1 N−D", "D Precision", "N Precision"]].round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
curve_names = ["Original (SVM, 27, default threshold)", "L1 (HistGB, set D, max-F1)",
               "E1 (HistGB, early signal, max-F1)", CHART_NEW, "R (HistGB, pre-diagnosis, max-F1)"]
colors = {"Original (SVM, 27, default threshold)": "gray", "L1 (HistGB, set D, max-F1)": "darkorange",
          "E1 (HistGB, early signal, max-F1)": "steelblue", CHART_NEW: "teal", "R (HistGB, pre-diagnosis, max-F1)": "slategray"}
for name in curve_names:
    feats, kind, rule = EVAL_SETS[name]
    p = cached_scores(feats, kind, "LOSO", rule)
    pr, rc, _ = precision_recall_curve(p.adhd, p.score)
    m = metrics(p.adhd, p.pred, p.score)
    ax.plot(rc, pr, color=colors[name], label=f"{name.split('(')[0].strip()} (PR-AUC {m['PR-AUC']:.3f})")
    ax.scatter(m["Recall"], m["Precision"], color=colors[name], edgecolor="black", zorder=3, s=40)
ax.axhline(kids.adhd.mean(), ls=":", color="gray")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–recall on LOSO scores (dots = max-F1 threshold)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG / "05_pr_curves.png", dpi=120)
plt.show()

def paired_bootstrap(name_a, name_b, n=1000):
    a = cached_scores(*EVAL_SETS[name_a][:2], "LOSO", EVAL_SETS[name_a][2])
    b = cached_scores(*EVAL_SETS[name_b][:2], "LOSO", EVAL_SETS[name_b][2]).loc[a.index]
    states = np.array(sorted(a.state.unique()))
    idx = {s: np.where(a.state.to_numpy() == s)[0] for s in states}
    ya, sa, pa = a.adhd.to_numpy(), a.score.to_numpy(), a.pred.to_numpy()
    sb, pb = b.score.to_numpy(), b.pred.to_numpy()
    rng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(n):
        rows_ = np.concatenate([idx[s] for s in rng.choice(states, len(states))])
        ma = metrics(ya[rows_], pa[rows_], sa[rows_])
        mb = metrics(ya[rows_], pb[rows_], sb[rows_])
        diffs.append([mb["F1"] - ma["F1"], mb["PR-AUC"] - ma["PR-AUC"], mb["Precision"] - ma["Precision"]])
    diffs = np.array(diffs)
    point_a, point_b = metrics(ya, pa, sa), metrics(ya, pb, sb)
    point = [point_b[k] - point_a[k] for k in ["F1", "PR-AUC", "Precision"]]
    return pd.DataFrame({"Change": point, "95% CI low": np.percentile(diffs, 2.5, axis=0),
                         "95% CI high": np.percentile(diffs, 97.5, axis=0)}, index=["F1", "PR-AUC", "Precision"])

boot_nd = paired_bootstrap("L1 (HistGB, set D, max-F1)", CHART_NEW)
boot_no = paired_bootstrap("Original (SVM, 27, default threshold)", CHART_NEW)
boot_e1 = paired_bootstrap("R (HistGB, pre-diagnosis, max-F1)", "E1 (HistGB, early signal, max-F1)")
boot = pd.concat({"N − set D": boot_nd, "N − original SVM": boot_no, "E1 − set R": boot_e1}, axis=0)
boot.to_csv(OUT / "paired_state_bootstrap.csv")
boot.round(3)


## 9. Does It Generalize for the Right Reasons?

**Method.** Fairness is an audit, not a tuning step. Sex, age group, income (federal poverty level), and race/ethnicity are scored on the LOSO predictions. Race and ethnicity are not model inputs. Calibration is the reliability curve and the Brier score of the gradient-boosting probabilities on LOSO scores. Permutation importance is fit on the 40 development states and scored on the 11 untouched states. SHAP is added on a sample of those states because it is installed.

Two negative controls use the development states only. The leakage check adds `k4q23` (medication for emotions, concentration, or behavior), a T5 item that is not in any real feature set, and measures the jump in PR-AUC. The shuffled-label check refits the selected feature set after permuting the training labels and scores the 11 untouched states. That PR-AUC should sit near the prevalence in those states.


In [ ]:
RACE = {1: "White", 2: "Black", 3: "AIAN", 4: "Asian", 5: "NHPI", 7: "Multiracial"}
fair_base = kids[["sc_sex", "sc_age_years", "sc_race_r", "sc_hispanic_r", "fpl_i1"]].copy()
fair_base["sex"] = fair_base.sc_sex.map({1: "Male", 2: "Female"})
fair_base["age group"] = pd.cut(fair_base.sc_age_years, [5, 8, 11, 14, 17], labels=["6–8", "9–11", "12–14", "15–17"]).astype(str)
fair_base["race/ethnicity"] = fair_base.sc_race_r.map(RACE).where(fair_base.sc_hispanic_r != 1, "Hispanic")
fair_base["income"] = pd.cut(fair_base.fpl_i1, [0, 100, 200, 400, 10000], right=False,
                             labels=["<100% FPL", "100–199% FPL", "200–399% FPL", "≥400% FPL"]).astype(str)
rng = np.random.default_rng(SEED)

def group_metrics(y, pred):
    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return np.array([p, r, 2 * p * r / max(p + r, 1e-12)])

def fairness(name):
    feats, kind, rule = EVAL_SETS[name]
    p = cached_scores(feats, kind, "LOSO", rule).join(fair_base)
    rows = []
    for col in ["sex", "age group", "income", "race/ethnicity"]:
        for g, d in p.groupby(col):
            y, pr = d.adhd.to_numpy(), d.pred.to_numpy()
            boots = np.vstack([group_metrics(y[i], pr[i]) for i in (rng.integers(0, len(d), len(d)) for _ in range(300))])
            lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
            est = group_metrics(y, pr)
            rows.append({"Config": name, "Attribute": col, "Group": g, "Children": len(d), "ADHD cases": int(y.sum()),
                         **{m: est[j] for j, m in enumerate(["Precision", "Recall", "F1"])},
                         **{f"{m} 95% CI": f"{lo[j]:.2f}–{hi[j]:.2f}" for j, m in enumerate(["Precision", "Recall", "F1"])}})
    return pd.DataFrame(rows)

fair = pd.concat([fairness(n) for n in ["L1 (HistGB, set D, max-F1)", CHART_NEW, "E1 (HistGB, early signal, max-F1)"]])
fair.to_csv(OUT / "fairness_results.csv", index=False)
fair[fair.Config == CHART_NEW].round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
brier = {}
for name, color in [("L1 (HistGB, set D, max-F1)", "darkorange"), (CHART_NEW, "teal"),
                    ("E1 (HistGB, early signal, max-F1)", "steelblue"), ("R (HistGB, pre-diagnosis, max-F1)", "slategray")]:
    feats, kind, rule = EVAL_SETS[name]
    p = cached_scores(feats, kind, "LOSO", rule)
    if kind == "svm_bal":
        continue
    frac, mean_pred = calibration_curve(p.adhd, p.score, n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac, marker="o", ms=3, color=color, label=name.split("(")[0].strip())
    brier[name] = brier_score_loss(p.adhd, p.score)
axes[0].plot([0, 1], [0, 1], ls="--", color="gray")
axes[0].set_xlabel("Predicted probability")
axes[0].set_ylabel("Observed ADHD rate")
axes[0].set_title("Calibration on LOSO scores")
axes[0].legend(fontsize=7)
pd.Series(brier, name="Brier").sort_values().plot.barh(ax=axes[1], color="teal")
axes[1].set_title("Brier score (lower is better)")
plt.tight_layout()
plt.savefig(FIG / "06_calibration.png", dpi=120)
plt.show()
pd.Series(brier, name="Brier score").round(4)


In [ ]:
def fit_explain(feats, kind, params_index=-1):
    params = MODEL_GRIDS[kind][params_index]
    model = make_pipeline(feats, make_model(kind, params)).fit(dev[columns_needed(feats)], dev.adhd)
    return model

explain_feats = N_SELECTED if N_SELECTED != FD else N_FULL
explain_label = "N selected" if N_SELECTED != FD else "N full (none kept; importance is descriptive only)"
model_n = fit_explain(explain_feats, "hgb")
model_d = fit_explain(FD, "hgb")
input_cols = columns_needed(explain_feats)
r = permutation_importance(model_n, holdout[input_cols], holdout.adhd,
                           scoring="average_precision", n_repeats=5, random_state=SEED, n_jobs=N_JOBS)
# Importance is on the columns the pipeline receives. Age-band z-scores are built inside the pipeline from these columns.
imp = pd.Series(r.importances_mean, index=input_cols).sort_values(ascending=False)
imp = imp.rename(index=lambda c: f"{c} — {str(DESC.get(c, catalog.loc[c, 'description']) if c in catalog.index else '')[:42]}")
imp.to_csv(OUT / "permutation_importance.csv", header=["mean_drop_in_PR_AUC"])
fig, ax = plt.subplots(figsize=(8, 6))
imp.head(15).sort_values().plot.barh(ax=ax, color="teal")
ax.set_xlabel("Drop in PR-AUC when the column is shuffled")
ax.set_title(f"Permutation importance on the 11 untouched states ({explain_label})")
plt.tight_layout()
plt.savefig(FIG / "07_permutation_importance.png", dpi=120)
plt.show()
imp.head(12).round(4)


In [ ]:
shap_note = "SHAP was not run."
if LIBS["shap"]:
    import shap
    rng_s = np.random.default_rng(SEED)
    take = rng_s.choice(holdout.index.to_numpy(), size=min(800, len(holdout)), replace=False)
    sample = holdout.loc[take]
    try:
        prep = model_n[:-1]
        transformed = prep.transform(sample[columns_needed(explain_feats)])
        if isinstance(transformed, pd.DataFrame):
            transformed = transformed.to_numpy()
        explainer = shap.TreeExplainer(model_n.named_steps["model"])
        sv = explainer.shap_values(transformed)
        if isinstance(sv, list):
            sv = sv[-1]
        mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=explain_feats).sort_values(ascending=False)
        mean_abs.rename(index=lambda c: f"{c} — {str(DESC.get(c, ''))[:42]}").head(15).sort_values().plot.barh(figsize=(8, 6), color="teal")
        plt.xlabel("Mean |SHAP| on 800 untouched-state children")
        plt.title("SHAP on the untouched states")
        plt.tight_layout()
        plt.savefig(FIG / "08_shap.png", dpi=120)
        plt.show()
        mean_abs.head(12).rename("mean_|SHAP|").to_csv(OUT / "shap_importance.csv")
        shap_note = "SHAP ran on 800 children from the 11 untouched states."
        display(mean_abs.head(12).round(4))
    except Exception as err:
        shap_note = f"SHAP is installed but failed ({type(err).__name__}: {err}). Permutation importance is the explanation that was saved."
        print(shap_note)
else:
    shap_note = "SHAP is not installed. Permutation importance is the explanation that was saved."
    print(shap_note)


In [ ]:
# Leakage sanity check and shuffled-label control. Both are scored only to audit; neither changes the feature set.
leak_feats = uniq(list(FD) + ["k4q23"])
if "k4q23" in FD or "k4q23" in N_FULL:
    raise RuntimeError("k4q23 leaked into a real feature set.")
s_d = OOF[("L1 set D", "hgb")]
s_leak = dev_oof(leak_feats, "hgb")
leak_delta = average_precision_score(y_dev, s_leak) - average_precision_score(y_dev, s_d)
print(f"Adding k4q23 (medication for emotions/concentration/behavior) to set D changes development PR-AUC by {leak_delta:+.3f}.")

rng_l = np.random.default_rng(SEED)
y_shuf = rng_l.permutation(dev.adhd.to_numpy())
shuf_model = make_pipeline(FD, make_model("hgb", MODEL_GRIDS["hgb"][-1])).fit(dev[columns_needed(FD)], y_shuf)
shuf_score = get_scores(shuf_model, holdout[columns_needed(FD)])
shuf_pr = average_precision_score(holdout.adhd, shuf_score)
print(f"Shuffled-label model on the 11 untouched states: PR-AUC {shuf_pr:.3f} (prevalence in those states {holdout.adhd.mean():.3f}).")
pd.DataFrame([{"check": "T5 medication added to set D", "Δ PR-AUC on development states": leak_delta},
              {"check": "shuffled labels, scored on 11 untouched states", "Δ PR-AUC on development states": shuf_pr}]).round(3)


## 10. Save the Model and the Experiment Settings

**Method.** The model that the keep rule selects is refit on all children. Its max-F1 threshold is chosen from state-grouped out-of-fold scores on all children, then the model is refit on everyone and saved. If no new family passed the rule, the saved model is set D, and the settings file says that.


In [ ]:
final_feats = N_SELECTED
final_kind = "hgb"
oof_all = []
for params in MODEL_GRIDS[final_kind]:
    s = cross_val_predict(make_pipeline(final_feats, make_model(final_kind, params)), kids[columns_needed(final_feats)],
                          kids.adhd, groups=kids.state, cv=GroupKFold(5), method="predict_proba", n_jobs=N_JOBS)
    oof_all.append(s[:, 1])
best_i = int(np.argmax([average_precision_score(kids.adhd, s) for s in oof_all]))
final_params = MODEL_GRIDS[final_kind][best_i]
final_threshold = choose_threshold(kids.adhd, oof_all[best_i], "max_f1")
final_model = make_pipeline(final_feats, make_model(final_kind, final_params)).fit(kids[columns_needed(final_feats)], kids.adhd)
joblib.dump({"model": final_model, "threshold": final_threshold, "features": final_feats, "rule": "max_f1"},
            MODELS_DIR / "better_features_model.joblib")
settings = {
    "seed": SEED,
    "untouched_test_states": TEST_STATES,
    "population": int(len(kids)),
    "selection_rule": "On the 40 development states, add a new family to set D only if ΔPR-AUC ≥ 0.005 in at least 4 of 5 state-grouped folds. Decided with HistGradientBoosting.",
    "families_kept": kept_families,
    "features_added": KEPT_COLS,
    "feature_set_L0": F27,
    "feature_set_L1_D": FD,
    "feature_set_R": FR,
    "feature_set_E1": E1,
    "feature_set_E2": E2,
    "feature_set_N": N_FULL,
    "feature_set_selected": final_feats,
    "final_model": final_kind,
    "final_params": final_params,
    "threshold_rule": "max_f1",
    "final_threshold": float(final_threshold),
    "libraries": LIBS,
    "shap_note": shap_note,
    "banned_inputs": sorted(BANNED_INPUTS),
    "adult2_note": "a2_grade, a2_menthealth, a2_physhealth, and a2_age are about 20% missing because there is no second caregiver. They fail the ≤10% rule and are not model inputs. no_adult2 keeps that skip.",
    "age_limited_not_used": "k6q70_r, k6q72_r, k6q73_r, distracted, and hoursleep05 are essentially missing at ages 6–17 and are not used. k6q71_r is asked at these ages and is eligible.",
}
(OUT / "experiment_settings.json").write_text(json.dumps(settings, indent=2, default=str), encoding="utf-8")
print("Saved", MODELS_DIR / "better_features_model.joblib", "threshold", round(float(final_threshold), 4), "params", final_params)
print(f"Total run time so far: {(time.time() - T0) / 60:.1f} minutes")


## 11. Conclusions

**Method.** The paragraphs, the slide table, and the five takeaways are written from the tables in this run. They are not typed in ahead of the results.


In [ ]:
loso_m = results[results.Protocol == "LOSO"].set_index("Config")
geo_m = results[results.Protocol == "Geographic holdout"].set_index("Config")
dev_h = show[show.model == "hgb"].set_index("set")

def loso_get(name, col):
    return float(loso_m.loc[name, col])

def fmt(x):
    return f"{x:.3f}"

lines = []
d_pr = float(dev_h.loc["L1 set D", "PR-AUC"])
n_pr = float(dev_h.loc["N full new set", "PR-AUC"])
e1_pr = float(dev_h.loc["E1 early signal", "PR-AUC"])
r_pr = float(dev_h.loc["R pre-diagnosis", "PR-AUC"])
raw_pr = float(dev_h.loc["N raw items only", "PR-AUC"])
lines.append("Result. On the 40 development states, histogram gradient boosting reached "
             f"PR-AUC {fmt(d_pr)} with set D, {fmt(n_pr)} with the full new set (Δ {n_pr - d_pr:+.3f}), "
             f"and {fmt(raw_pr)} with the same items and no engineered columns (engineered − raw {n_pr - raw_pr:+.3f}). "
             f"The early-signal set E1 reached {fmt(e1_pr)} against {fmt(r_pr)} for the old pre-diagnosis set R (Δ {e1_pr - r_pr:+.3f}).")
if kept_families:
    lines.append("The keep rule (Δ PR-AUC ≥ 0.005 in at least 4 of 5 development folds) kept: " + ", ".join(kept_families) + ".")
else:
    lines.append("The keep rule kept no new family. No added family raised PR-AUC by 0.005 in at least 4 of the 5 development folds.")
hgb_add_s = add_table[add_table.model == "hgb"].sort_values("Δ PR-AUC", ascending=False)
top = hgb_add_s.iloc[0]
lines.append(f"The largest single-family change on top of set D was {top.family} ({top['Δ PR-AUC']:+.3f} PR-AUC, "
             f"gain of at least 0.005 in {int(top['folds ≥ 0.005'])} of 5 folds).")

name_n = CHART_NEW
lines.append("Under leave-one-state-out, the original SVM on 27 features has precision "
             f"{fmt(loso_get('Original (SVM, 27, default threshold)', 'Precision'))}, recall {fmt(loso_get('Original (SVM, 27, default threshold)', 'Recall'))}, "
             f"F1 {fmt(loso_get('Original (SVM, 27, default threshold)', 'F1'))}, PR-AUC {fmt(loso_get('Original (SVM, 27, default threshold)', 'PR-AUC'))}. "
             f"Set D has precision {fmt(loso_get('L1 (HistGB, set D, max-F1)', 'Precision'))}, recall {fmt(loso_get('L1 (HistGB, set D, max-F1)', 'Recall'))}, "
             f"F1 {fmt(loso_get('L1 (HistGB, set D, max-F1)', 'F1'))}, PR-AUC {fmt(loso_get('L1 (HistGB, set D, max-F1)', 'PR-AUC'))}, "
             f"and precision at recall 0.70 of {fmt(loso_get('L1 (HistGB, set D, max-F1)', 'Precision at recall 0.70'))}. "
             f"The full new set has precision {fmt(loso_get(name_n, 'Precision'))}, recall {fmt(loso_get(name_n, 'Recall'))}, "
             f"F1 {fmt(loso_get(name_n, 'F1'))}, PR-AUC {fmt(loso_get(name_n, 'PR-AUC'))}, "
             f"and precision at recall 0.70 of {fmt(loso_get(name_n, 'Precision at recall 0.70'))}.")
lines.append("The paired state bootstrap for the full new set minus set D is "
             f"ΔF1 {boot_nd.loc['F1', 'Change']:+.3f} (95% CI {boot_nd.loc['F1', '95% CI low']:+.3f} to {boot_nd.loc['F1', '95% CI high']:+.3f}) and "
             f"ΔPR-AUC {boot_nd.loc['PR-AUC', 'Change']:+.3f} (95% CI {boot_nd.loc['PR-AUC', '95% CI low']:+.3f} to {boot_nd.loc['PR-AUC', '95% CI high']:+.3f}).")
lines.append("Early signal versus the old set R, on LOSO: "
             f"E1 F1 {fmt(loso_get('E1 (HistGB, early signal, max-F1)', 'F1'))} and PR-AUC {fmt(loso_get('E1 (HistGB, early signal, max-F1)', 'PR-AUC'))}, "
             f"against R F1 {fmt(loso_get('R (HistGB, pre-diagnosis, max-F1)', 'F1'))} and PR-AUC {fmt(loso_get('R (HistGB, pre-diagnosis, max-F1)', 'PR-AUC'))}. "
             f"Paired bootstrap ΔF1 {boot_e1.loc['F1', 'Change']:+.3f} (95% CI {boot_e1.loc['F1', '95% CI low']:+.3f} to {boot_e1.loc['F1', '95% CI high']:+.3f}).")
lines.append(f"Leakage check: adding k4q23 to set D changes development PR-AUC by {leak_delta:+.3f}. "
             f"Shuffled-label PR-AUC on the 11 untouched states is {shuf_pr:.3f}, next to a prevalence of {holdout.adhd.mean():.3f}.")
lines.append("Interpretation. The label is a parent's report that a doctor ever said the child has ADHD, not a record review. "
             "T3 behavior items and T4 diagnoses can be consequences of that diagnosis or part of the same evaluation, so a high score on N or on set D is partly recognition of a child who was already identified. "
             "E1 is the set that avoids those items. Parent aggravation and current health are in E1 because the questions are not behavior checklists, but they can still respond to the child's difficulties. "
             "Precision remains limited by prevalence, and it is lowest in states with less reported ADHD, including Nevada, New Mexico, California, Hawaii, and Wyoming. "
             "This is a research model of survey answers. It is not a diagnostic tool and it should not be used to screen children.")

# Slide table, LOSO, max-F1 operating point plus the threshold-free precision at recall 0.70.
slide_rows = []
for label, key in [("Original SVM, 27 features", "Original (SVM, 27, default threshold)"),
                   ("Set D, gradient boosting", "L1 (HistGB, set D, max-F1)"),
                   ("Early signal (T0–T2)", "E1 (HistGB, early signal, max-F1)"),
                   ("Behavior, no other diagnoses", "E2 (HistGB, behavior no T4, max-F1)"),
                   ("Full new feature set", "N (HistGB, full new set, max-F1)"),
                   ("Old pre-diagnosis set R", "R (HistGB, pre-diagnosis, max-F1)")]:
    slide_rows.append({"Model": label, "Precision": loso_get(key, "Precision"), "Recall": loso_get(key, "Recall"),
                       "F1": loso_get(key, "F1"), "PR-AUC": loso_get(key, "PR-AUC"),
                       "Precision at recall 0.70": loso_get(key, "Precision at recall 0.70")})
slide = pd.DataFrame(slide_rows).set_index("Model")
slide.to_csv(OUT / "slide_summary.csv")

gap = n_pr - d_pr
if gap >= 0.005 and kept_families:
    t1 = f"New features cleared the 0.005 bar: {', '.join(kept_families)} added {gap:.3f} PR-AUC on the development states over set D."
elif gap >= 0.005:
    t1 = f"The full bundle gained {gap:.3f} PR-AUC over set D on the development states, but no single new family cleared the 4-of-5-folds rule."
else:
    t1 = f"Better features did not beat set D. The full new set changed development PR-AUC by {gap:+.3f}, short of the 0.005 rule."
e1_loso_f1 = loso_get("E1 (HistGB, early signal, max-F1)", "F1")
r_loso_f1 = loso_get("R (HistGB, pre-diagnosis, max-F1)", "F1")
t2 = f"The early-signal set {'beats' if e1_loso_f1 > r_loso_f1 + 0.01 else 'does not meaningfully beat'} the old pre-diagnosis set on unseen states (LOSO F1 {e1_loso_f1:.3f} versus {r_loso_f1:.3f})."
t3 = f"The biggest family added to set D was {top.family} ({top['Δ PR-AUC']:+.3f} PR-AUC)."
t4 = (f"On unseen states the full new set and set D differ by ΔF1 {boot_nd.loc['F1', 'Change']:+.3f} "
      f"(95% CI {boot_nd.loc['F1', '95% CI low']:+.3f} to {boot_nd.loc['F1', '95% CI high']:+.3f}).")
t5 = "What still limits the model is the parent-reported label, the chance that behavior and other diagnoses come after the ADHD label, and precision that falls in low-prevalence states. This is not a diagnostic tool."
takeaways = [t1, t2, t3, t4, t5]
(OUT / "conclusions.txt").write_text("\n\n".join(lines) + "\n\n" + "\n".join(f"{i}. {t}" for i, t in enumerate(takeaways, 1)), encoding="utf-8")
print("\n\n".join(lines))
print("\nSlide table (LOSO):")
display(slide.round(3))
print("\nFive takeaways:")
for i, t in enumerate(takeaways, 1):
    print(f"{i}. {t}")
print(f"\nTotal notebook run time: {(time.time() - T0) / 60:.1f} minutes")
print("Files:", sorted(p.name for p in OUT.iterdir()))


## Finished unseen-state results

These numbers are from output/nsch_features/results_all_protocols.csv, written after the nested runs completed. Set D matches the Improved notebook.

**Leave-one-state-out**

| Model | Precision | Recall | F1 | PR-AUC | Precision at recall 0.70 |
|---|---:|---:|---:|---:|---:|
| Original SVM, 27 features | 0.459 | 0.738 | 0.566 | 0.591 | 0.485 |
| Improved set D | 0.602 | 0.663 | 0.631 | 0.665 | 0.575 |
| Set D + memorycond | 0.637 | 0.700 | 0.667 | 0.708 | 0.637 |
| Early signal (T0–T2) | 0.426 | 0.572 | 0.488 | 0.482 | 0.357 |
| Old pre-diagnosis set R | 0.268 | 0.581 | 0.367 | 0.304 | 0.244 |

Adding memorycond raises LOSO F1 by 0.036 (95% state-bootstrap interval 0.033 to 0.040) and PR-AUC by 0.043 (0.039 to 0.046). The early-signal set beats set R (F1 0.488 versus 0.367) and stays well below set D.

Sleep, parent stress, neighborhood, hardship, the ACE count, adult 2, and the engineered scores did not clear the pre-set rule of a 0.005 PR-AUC gain in at least 4 of 5 development folds. memorycond asks whether the child currently has serious difficulty concentrating, remembering, or making decisions, so the gain is recognition of a current difficulty. The label is a parent's report. This is not a diagnostic tool.


## 12. Linear regression, logistic regression, and a decision tree

**Method.** The same children, the same 11 untouched states, and the same six protocols are used. Two feature sets are scored: improved set D, and set D plus `memorycond` (the only new family that passed the keep rule).

* **Linear regression** predicts the 0/1 label directly. The score is that prediction, and the max-F1 threshold is chosen on inner validation scores.
* **Logistic regression** tries C = 0.1 and C = 1 inside the training states and keeps the one with higher PR-AUC.
* **Decision tree** uses the original project's grid: depth 6 or 8, and at least 50 or 200 children per leaf, with balanced class weights. The inner state-grouped folds pick the setting and the max-F1 threshold.

Preprocessing matches the earlier logistic pipeline: median impute, one-hot codes, and scaled numbers, all fit on the training fold only. Results are written to `output/nsch_features/classical_models_results.csv`.


In [ ]:
# CLASSICAL MODELS
# Fitted by run_classical.py on the same splits. This cell displays the saved table.
from pathlib import Path
import pandas as pd
path = Path("output/nsch_features/classical_models_results.csv")
classical = pd.read_csv(path)
loso = classical[classical.Protocol == "LOSO"].set_index(["Model", "Features"])
cols = ["Precision", "Recall", "F1", "PR-AUC", "Precision at recall 0.70"]
loso[cols].round(3)
